# Imports

In [ ]:
import pathlib
import tqdm
import pandas as pd
import numpy as np

from experiments.models import (
    LightGBMForecast,
    LightGBMARForecast,
    LightGBMSTLForecast,
)

from experiments.utils import load_experiments_specs

# Load Experiment Specifications

In [ ]:
dataset = "ABC"

In [ ]:
# Load specifications
train_type = "local"
experiment_specs = load_experiments_specs(
    dataset=dataset,
    train_type=train_type,
)

# Train and Test Data
train_df = experiment_specs["train"]
test_df = experiment_specs["test"]
data_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# Meta Data
meta = experiment_specs["meta"]
features = meta["features"]
fcst_h = meta["fcst_h"]
freq = meta["freq"]
lags = meta["lags"]
max_lag = meta["max_lag"]
series_ids = meta["series_ids"]
seed = 123

# Hyper-Parameters
config = experiment_specs["config"]
params_lgb = config["lgb_params"]
params_lgb_stl = {**params_lgb, **config["lgb_stl_params"]}

# Run Models

In [ ]:
np.random.seed(seed)

model_forecasts = []

for series in tqdm.tqdm(series_ids, total=len(series_ids)):

    ###
    # Subset Data
    ###
    series_df = data_df[data_df["series_id"] == series].reset_index(drop=True).copy()

    ###
    # Split Data into Train-Test
    ###
    test = series_df.tail(fcst_h)
    train = series_df.drop(test.index)

    ###
    # LightGBM-AR
    ###
    lgb_ar_fcst = LightGBMARForecast(
        params_lgb,
        train,
        test.drop(columns=["value"]),
        features,
        freq,
        fcst_h,
        lags,
        max_lag,
        n_series=1
    )

    ###
    # LightGBM
    ###
    if params_lgb["use_time_index"]:
        lgb_feats = meta["features"] + ["time_idx"]
    else:
        lgb_feats = meta["features"]

    lgb_fcst = LightGBMForecast(
        params_lgb,
        train,
        test.drop(columns=["value"]),
        lgb_feats
    )

    ###
    # LightGBM-STL
    ###
    lgb_stl_fcst = LightGBMSTLForecast(
        params_lgb_stl,
        train,
        test.drop(columns=["value"]),
        lgb_feats,
        fcst_h,
        lags
    )

    ###
    # Append Forecasts
    ###
    model_forecasts.append(
        pd.concat(
            [
                lgb_ar_fcst,
                lgb_fcst,
                lgb_stl_fcst,
            ],
            axis=0
        )
    )

# Combine Forecasts

In [ ]:
# Concatenate all forecasts into a single DataFrame
fcsts_df = pd.concat([
    pd.concat(model_forecasts, axis=0),
], axis=0).reset_index(drop=True)

# Add actual values and dataset information
fcsts_df = pd.merge(
    fcsts_df,
    test_df[["dataset", "series_id", "date", "value"]],
    on=["series_id", "date"],
    how="inner"
)

# Save Forecasts

In [ ]:
repo_root = pathlib.Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

result_path = repo_root / "experiments" / "runs" / "results" / train_type
result_path.mkdir(parents=True, exist_ok=True)

fcsts_df.to_csv(result_path / f"{dataset.lower()}_lgbm_fcsts.csv", index=False)